# Hand Character Recognition - Training Notebook

CNN untuk pengenalan tulisan tangan **0-9 dan A-Z** (36 karakter total).

**Dataset:**
- [MNIST](https://www.kaggle.com/datasets/hojjatk/mnist-dataset) — digits 0-9
- [EMNIST](https://www.kaggle.com/datasets/crawford/emnist) — letters A-Z

**Compatible:** Google Colab (GPU recommended) • VS Code Jupyter • Jupyter Notebook lokal

## 1. Install & Import

In [ ]:
!pip install -q tensorflow kagglehub matplotlib seaborn scikit-learn

In [ ]:
import os
import struct
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import kagglehub

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Helper Functions

In [ ]:
def load_idx_images(filename):
    with open(filename, 'rb') as f:
        magic, num, rows, cols = struct.unpack('>IIII', f.read(16))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows, cols)

def load_idx_labels(filename):
    with open(filename, 'rb') as f:
        magic, num = struct.unpack('>II', f.read(8))
        return np.frombuffer(f.read(), dtype=np.uint8)

def find_file(folder, must_contain, must_not_contain=None):
    must_not_contain = must_not_contain or []
    for root, dirs, files in os.walk(folder):
        for f in files:
            fl = f.lower()
            if all(p in fl for p in must_contain) and not any(n in fl for n in must_not_contain):
                return os.path.join(root, f)
    raise FileNotFoundError(f'Tidak ketemu: {must_contain}')

def index_to_char(idx):
    if 0 <= idx <= 9:
        return str(idx)
    return chr(ord('A') + idx - 10)

## 3. Download Dataset MNIST (Digits)

In [ ]:
mnist_path = kagglehub.dataset_download('hojjatk/mnist-dataset')
print(f'MNIST: {mnist_path}')

x_d_train = load_idx_images(find_file(mnist_path, ['train', 'image']))
y_d_train = load_idx_labels(find_file(mnist_path, ['train', 'label']))
x_d_test  = load_idx_images(find_file(mnist_path, ['t10k', 'image']))
y_d_test  = load_idx_labels(find_file(mnist_path, ['t10k', 'label']))

print(f'Train: {x_d_train.shape}, Test: {x_d_test.shape}')

## 4. Download Dataset EMNIST (Letters A-Z)

EMNIST dataset cukup besar (~500MB), download sekitar 1-2 menit di Colab.

**Catatan teknis:** Image EMNIST formatnya rotated 90° + mirror, jadi perlu di-transpose. Labels juga dimulai dari 1 (A=1, B=2, ..., Z=26), kita ubah ke 0-25 lalu shift ke 10-35 saat digabung.

In [ ]:
emnist_path = kagglehub.dataset_download('crawford/emnist')
print(f'EMNIST: {emnist_path}')

x_l_train = load_idx_images(find_file(emnist_path, ['letters', 'train', 'image'], ['mapping']))
y_l_train = load_idx_labels(find_file(emnist_path, ['letters', 'train', 'label'], ['mapping']))
x_l_test  = load_idx_images(find_file(emnist_path, ['letters', 'test', 'image'], ['mapping']))
y_l_test  = load_idx_labels(find_file(emnist_path, ['letters', 'test', 'label'], ['mapping']))

# Transpose untuk fix rotation/mirror EMNIST
x_l_train = np.transpose(x_l_train, (0, 2, 1))
x_l_test = np.transpose(x_l_test, (0, 2, 1))

# Labels EMNIST letters: A=1...Z=26 → ubah ke 0-25
y_l_train = y_l_train - 1
y_l_test = y_l_test - 1

print(f'Train: {x_l_train.shape}, Test: {x_l_test.shape}')
print(f'Letter classes: {len(np.unique(y_l_train))}')

## 5. Combine MNIST + EMNIST → 36 Classes

- Class 0-9 : digits
- Class 10-35 : letters A-Z

In [ ]:
# Shift letter labels 0-25 → 10-35
y_l_train = y_l_train + 10
y_l_test = y_l_test + 10

x_train = np.concatenate([x_d_train, x_l_train], axis=0)
y_train = np.concatenate([y_d_train, y_l_train], axis=0)
x_test = np.concatenate([x_d_test, x_l_test], axis=0)
y_test = np.concatenate([y_d_test, y_l_test], axis=0)

# Shuffle training
rng = np.random.default_rng(42)
idx = rng.permutation(len(x_train))
x_train, y_train = x_train[idx], y_train[idx]

print(f'Combined Train: {x_train.shape}')
print(f'Combined Test:  {x_test.shape}')
print(f'Total classes: {len(np.unique(y_train))} (should be 36)')

## 6. Visualisasi Sample

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    sample_idx = i * 1000
    ax.imshow(x_train[sample_idx], cmap='gray')
    ax.set_title(f'"{index_to_char(y_train[sample_idx])}" (cls {y_train[sample_idx]})')
    ax.axis('off')
plt.suptitle('Sample dari combined dataset')
plt.tight_layout()
plt.show()

## 7. Preprocessing

In [ ]:
NUM_CLASSES = 36

x_train = x_train.astype('float32').reshape(-1, 28, 28, 1) / 255.0
x_test  = x_test.astype('float32').reshape(-1, 28, 28, 1) / 255.0
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat  = to_categorical(y_test, NUM_CLASSES)

print(f'x_train: {x_train.shape}, y_train_cat: {y_train_cat.shape}')

## 8. Build CNN (untuk 36 classes)

In [ ]:
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 9. Training

Dataset gabungan ~880K sample. Di Colab GPU: ~3-5 menit/epoch. Total ~30-50 menit untuk 10 epochs.

Kalau mau lebih cepat untuk testing, set `EPOCHS = 5`.

In [ ]:
EPOCHS = 10
BATCH_SIZE = 256  # batch lebih besar karena dataset gede

datagen = ImageDataGenerator(
    rotation_range=10, zoom_range=0.1,
    width_shift_range=0.1, height_shift_range=0.1
)
datagen.fit(x_train)

history = model.fit(
    datagen.flow(x_train, y_train_cat, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(x_test, y_test_cat),
    verbose=1
)

## 10. Evaluasi

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f'Test accuracy: {test_acc*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout()
plt.show()

## 11. Per-class Accuracy

Akurasi tiap karakter (digits biasanya >99%, beberapa huruf yang mirip seperti I/L/1, O/0, S/5 mungkin lebih rendah).

In [ ]:
y_pred = np.argmax(model.predict(x_test), axis=1)

print(f'{"Class":<8} {"Char":<6} {"Accuracy":<10}')
print('-' * 30)
for i in range(NUM_CLASSES):
    mask = y_test == i
    if mask.sum() == 0:
        continue
    acc = (y_pred[mask] == i).sum() / mask.sum()
    print(f'{i:<8} {index_to_char(i):<6} {acc*100:>6.2f}%')

## 12. Save Model

In [ ]:
os.makedirs('../models', exist_ok=True)
model.save('../models/char_model.keras')
model.save('../models/char_model.h5')
print('Saved: ../models/char_model.keras dan .h5')

In [ ]:
# Khusus Colab: download model ke laptop
# Uncomment baris di bawah jika kamu di Colab dan mau download:

# from google.colab import files
# files.download('../models/char_model.keras')

## Selesai!

Model sudah siap. Selanjutnya di laptop:
1. Taruh `char_model.keras` di folder `models/`
2. Install: `pip install scipy tensorflow pillow numpy`
3. Jalankan: `python interface/draw_app.py`
4. Pilih mode (Digits / Letters / Mixed) sesuai kebutuhan, lalu tulis di kanvas